# Loading and handling data

[Download notebook](notebooks/07_handling_data.ipynb).

In this chapter we learn how to load data from various sources and how
to handle tabular data using `pandas`.

## Loading data from files (`csv`, `xlsx`, `json`)

Data often comes as files. The most common formats are CSV
(comma-separated values), Excel (`.xlsx`), and JSON (JavaScript Object
Notation). Python has built-in support for CSV and JSON, and `pandas`
can read all three. The built-in modules are:

- `csv.reader(file)`: reads a CSV file row by row (each row is a list of
  strings).
- `csv.DictReader(file)`: reads a CSV file row by row (each row is a
  dictionary with column names as keys).
- `json.loads(s)`: parses a JSON string into a Python object (dict,
  list, etc.).
- `json.dumps(obj)`: converts a Python object to a JSON string.
- `json.load(file)` / `json.dump(obj, file)`: read/write JSON from/to a
  file.

In [1]:
import csv
import io

# We simulate a CSV file using a string
csv_string = """name,age,city
Alice,30,Berlin
Bob,25,Munich
Charlie,35,Hamburg"""

reader = csv.DictReader(io.StringIO(csv_string))
for row in reader:
    print(row)

{'name': 'Alice', 'age': '30', 'city': 'Berlin'}
{'name': 'Bob', 'age': '25', 'city': 'Munich'}
{'name': 'Charlie', 'age': '35', 'city': 'Hamburg'}

In [2]:
import json

# parsing a JSON string
json_string = '{"name": "Alice", "age": 30, "scores": [85, 92, 78]}'
data = json.loads(json_string)
print(data)
print("name:", data["name"])

{'name': 'Alice', 'age': 30, 'scores': [85, 92, 78]}
name: Alice

In [3]:
# converting back to JSON
person = {"name": "Bob", "age": 25, "city": "Munich"}
print(json.dumps(person, indent=2))

{
  "name": "Bob",
  "age": 25,
  "city": "Munich"
}

In practice, you will usually use `pandas` to read data files (see
below). For Excel files, `pd.read_excel("data.xlsx")` requires the
`openpyxl` package.

## Loading data from online resources (`requests`)

The `requests` library allows downloading data from the internet.

- `requests.get(url)`: sends a GET request and returns a response
  object.
- `response.status_code`: HTTP status code (200 means success).
- `response.text`: the response body as a string.
- `response.json()`: parse the response body as JSON.

In [4]:
import requests

url = "https://jsonplaceholder.typicode.com/todos/1"
response = requests.get(url)
print("status code:", response.status_code)
print("content:", response.json())

status code: 200
content: {'userId': 1, 'id': 1, 'title': 'delectus aut autem', 'completed': False}

You can also combine `requests` with `pandas`:
`pd.read_csv("https://example.com/data.csv")` reads a CSV file directly
from a URL.

## Dataframes (`pandas`)

`pandas` is the standard library for handling tabular data in Python.
Its central data structure is the `DataFrame`, which can be thought of
as a matrix together with row and column labels.

In [5]:
import pandas as pd
import numpy as np

Here, `df` is a `DataFrame`:

- `pd.DataFrame(dict)`: create a DataFrame from a dictionary (keys
  become column names).
- `df["col"]`: select a single column (returns a `Series`).
- `df[["col1", "col2"]]`: select multiple columns.
- `df.iloc[i]`: select row `i` by position.
- `df.loc[i, "col"]`: select by label.
- `df[df["col"] > value]`: filter rows by a condition.
- `df.head(n)` / `df.tail(n)`: first / last `n` rows.
- `df.describe()`: summary statistics for all numeric columns.
- `df.dtypes`: data types of all columns.
- `df.info()`: concise summary of the DataFrame.
- `df.sort_values("col")`: sort by a column.
- `df.groupby("col").mean()`: group by a column and compute means.
- `df.isna()`: boolean mask of missing values.
- `df.dropna()`: drop rows with missing values.
- `df.fillna(value)`: fill missing values.
- `df.drop(columns=["col"])`: remove a column.
- `df.to_numpy()`: convert to a NumPy array.
- `pd.read_csv(path)`: read a CSV file into a DataFrame.
- `pd.to_datetime(...)`: convert strings to datetime objects.

In [6]:
df = pd.DataFrame({
    "age": [22, 27, 31, 45, 38, 29],
    "income": [42000, 51000, 64000, 83000, 72000, 56000],
    "hours_studied": [10, 12, 7, 4, 6, 9],
    "passed": [1, 1, 0, 0, 1, 1]
})
df

In [7]:
# selecting columns
print(df["age"])

0    22
1    27
2    31
3    45
4    38
5    29
Name: age, dtype: int64

In [8]:
print(df[["age", "income"]])

   age  income
0   22   42000
1   27   51000
2   31   64000
3   45   83000
4   38   72000
5   29   56000

In [9]:
# selecting rows
print(df.iloc[0])

age                 22
income           42000
hours_studied       10
passed               1
Name: 0, dtype: int64

In [10]:
print(df.loc[0:2, ["age", "passed"]])

   age  passed
0   22       1
1   27       1
2   31       0

In [11]:
# filtering
filtered = df[df["income"] > 55000]
filtered

In [12]:
# summary statistics
print(df.describe())

             age        income  hours_studied    passed
count   6.000000      6.000000       6.000000  6.000000
mean   32.000000  61333.333333       8.000000  0.666667
std     8.246211  14827.901627       2.898275  0.516398
min    22.000000  42000.000000       4.000000  0.000000
25%    27.500000  52250.000000       6.250000  0.250000
50%    30.000000  60000.000000       8.000000  1.000000
75%    36.250000  70000.000000       9.750000  1.000000
max    45.000000  83000.000000      12.000000  1.000000

In [13]:
# sorting
df_sorted = df.sort_values("income", ascending=False)
print(df_sorted)

   age  income  hours_studied  passed
3   45   83000              4       0
4   38   72000              6       1
2   31   64000              7       0
5   29   56000              9       1
1   27   51000             12       1
0   22   42000             10       1

In [14]:
# groupby
grouped = df.groupby("passed").mean(numeric_only=True)
print(grouped)

         age   income  hours_studied
passed                              
0       38.0  73500.0           5.50
1       29.0  55250.0           9.25

In [15]:
# missing values
df_missing = df.copy()
df_missing.loc[2, "income"] = np.nan
df_missing.loc[4, "hours_studied"] = np.nan

print(df_missing)
print()
print(df_missing.isna().sum())
print()
print(df_missing.dropna())

   age   income  hours_studied  passed
0   22  42000.0           10.0       1
1   27  51000.0           12.0       1
2   31      NaN            7.0       0
3   45  83000.0            4.0       0
4   38  72000.0            NaN       1
5   29  56000.0            9.0       1

age              0
income           1
hours_studied    1
passed           0
dtype: int64

   age   income  hours_studied  passed
0   22  42000.0           10.0       1
1   27  51000.0           12.0       1
3   45  83000.0            4.0       0
5   29  56000.0            9.0       1

In [16]:
# fill missing values with column means
df_filled = df_missing.fillna(df_missing.mean(numeric_only=True))
print(df_filled)

   age   income  hours_studied  passed
0   22  42000.0           10.0       1
1   27  51000.0           12.0       1
2   31  60800.0            7.0       0
3   45  83000.0            4.0       0
4   38  72000.0            8.4       1
5   29  56000.0            9.0       1

In [17]:
# from pandas to numpy
clean = df_missing.dropna()
X = clean[["age", "income", "hours_studied"]].to_numpy(dtype=float)
y = clean["passed"].to_numpy(dtype=float)
print("X.shape =", X.shape)
print("y =", y)

X.shape = (4, 3)
y = [1. 1. 0. 1.]

The guiding principle is: **pandas handles labeled tabular data; numpy
performs the mathematical core.**

In [18]:
# reading CSV files (here with a fallback if the file does not exist)
from pathlib import Path

csv_path = Path("ml_dataset.csv")

if csv_path.exists():
    df_csv = pd.read_csv(csv_path)
else:
    rng = np.random.default_rng(42)
    n = 120
    df_csv = pd.DataFrame({
        "age": rng.integers(18, 65, size=n),
        "income": rng.normal(55000, 14000, size=n),
        "hours_studied": rng.normal(8, 2, size=n),
    })
    logits = (
        -4
        + 0.00005 * df_csv["income"]
        + 0.25 * df_csv["hours_studied"]
        - 0.02 * df_csv["age"]
    )
    probs = 1 / (1 + np.exp(-logits))
    df_csv["target"] = (rng.random(n) < probs).astype(int)

df_csv.head()

In [19]:
print(df_csv.describe())

             age        income  hours_studied      target
count  120.00000    120.000000     120.000000  120.000000
mean    41.95000  53343.380771       7.991499    0.458333
std     12.83499  12735.834218       2.016080    0.500350
min     18.00000  25151.352070       2.866683    0.000000
25%     31.00000  43345.030963       6.670809    0.000000
50%     42.00000  52550.425867       7.808451    0.000000
75%     54.00000  61287.290680       8.879359    1.000000
max     63.00000  95794.074524      13.810134    1.000000

### Working with dates and times

`pandas` can parse and work with dates.

- `pd.to_datetime(...)`: convert strings to datetime.
- `dt.strftime(format)`: format a datetime as a string.

In [20]:
dates = pd.to_datetime(["2024-01-15", "2024-03-22", "2024-07-01"])
print(dates)
print("Month:", dates.month.tolist())
print("Day of week:", dates.day_of_week.tolist())

DatetimeIndex(['2024-01-15', '2024-03-22', '2024-07-01'], dtype='datetime64[us]', freq=None)
Month: [1, 3, 7]
Day of week: [0, 4, 0]

In [21]:
dt = pd.Timestamp("2024-06-15 14:30:00")
print(dt.strftime("%-m/%-d/%y, %H:%M"))

6/15/24, 14:30

## SQL databases (`sqlite3`)

Python has built-in support for SQLite databases via the `sqlite3`
module. This is useful for working with structured data that lives in a
database.

- `sqlite3.connect(path)`: connect to a database (use `":memory:"` for
  an in-memory database).
- `cursor.execute(sql)`: execute an SQL statement.
- `cursor.fetchall()`: fetch all results.
- `pd.read_sql_query(sql, conn)`: run an SQL query and return the result
  as a DataFrame.

In [22]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE students (
        name TEXT,
        age INTEGER,
        grade REAL
    )
""")

students = [
    ("Alice", 22, 1.3),
    ("Bob", 25, 2.0),
    ("Charlie", 23, 1.7),
    ("Diana", 27, 2.3),
]

cursor.executemany("INSERT INTO students VALUES (?, ?, ?)", students)
conn.commit()

In [23]:
# querying
cursor.execute("SELECT * FROM students WHERE age > 23")
for row in cursor.fetchall():
    print(row)

('Bob', 25, 2.0)
('Diana', 27, 2.3)

In [24]:
# reading SQL results into a pandas DataFrame
df_sql = pd.read_sql_query("SELECT * FROM students", conn)
print(df_sql)
conn.close()

      name  age  grade
0    Alice   22    1.3
1      Bob   25    2.0
2  Charlie   23    1.7
3    Diana   27    2.3